In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import gc

In [ ]:
# clearing GPU cache:
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# force garbage collection:
gc.collect()
print('cache cleared and garbage collected!')

cache cleared and garbage collected!


In [ ]:
proj_path = "/content/drive/MyDrive/llm_from_scratch/src"
data_path = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [ ]:
import os, sys
sys.path.append(proj_path)
sys.path.append(data_path)
os.chdir(proj_path)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/src


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [ ]:
from loading_weights import gpt as model
print("model loaded successfully.")

File already exists and is up-to-date: gpt2/774M/checkpoint
File already exists and is up-to-date: gpt2/774M/encoder.json
File already exists and is up-to-date: gpt2/774M/hparams.json
File already exists and is up-to-date: gpt2/774M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/774M/model.ckpt.index
File already exists and is up-to-date: gpt2/774M/model.ckpt.meta
File already exists and is up-to-date: gpt2/774M/vocab.bpe
model loaded successfully.


In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
import tiktoken, json
from torch.utils.data import Dataset
from transformers import TrainingArguments, Trainer

In [ ]:
class JsonInstructionDataset(Dataset):
    def __init__(self, json_data, max_length=512,):
        self.encoding = tiktoken.get_encoding('gpt2')
        self.json_data = json_data
        self.max_length = max_length

    def __len__(self):
        return len(self.json_data)

    def __getitem__(self, idx):
        item = self.json_data[idx]

        text = f"### Instruction:\n{item['instruction']}\n\n### Input:\n{item['input']}\n\n### Response:\n{item['output']}"

        # tokenizing with tiktoken
        tokens = self.encoding.encode(text)

        # truncating if necessary
        if len(tokens) > self.max_length:
            tokens = tokens[:self.max_length]

        input_ids = tokens

        # padding if necessary
        if len(input_ids) < self.max_length:
            padding_length = self.max_length - len(input_ids)
            padding = [self.encoding.eot_token] * padding_length
            input_ids = input_ids + padding

        return {
            'input_ids': torch.tensor(input_ids, dtype=torch.long),
            'labels': torch.tensor(input_ids, dtype=torch.long)
        }

In [ ]:
import json

In [ ]:
with open("/content/drive/MyDrive/llm_from_scratch/datasets/alpaca_gpt4_data.json", "r", encoding="utf-8") as f:
    json_data_gpt4 = json.load(f)

In [ ]:
print(len(json_data_gpt4))

52002


In [ ]:
json_gpt4_train, json_gpt4_val = json_data_gpt4[:5000], json_data_gpt4[5000:6000]
print(len(json_gpt4_train))
print(len(json_gpt4_val))

5000
1000


In [ ]:
print(json_gpt4_train[-1])
print(json_gpt4_val[-2])

{'instruction': 'Generate a list of 10 natural numbers between given two numbers.', 'input': 'Numbers: 5, 25', 'output': 'Here is a list of 10 natural numbers between 5 and 25: 6, 7, 8, 9, 10, 11, 12, 13, 14, 15.'}
{'instruction': 'Name at least three types of clouds.', 'input': '', 'output': 'There are many types of clouds, but here are three common ones: \n1. Cumulus: These are the fluffy, cotton-like clouds often seen in the sky on sunny days. \n2. Stratus: These are low, uniform layers of clouds that often result in overcast skies and light precipitation. \n3. Cirrus: These are thin, wispy, high-altitude clouds made of ice crystals.'}


In [ ]:
train_dataset = JsonInstructionDataset(json_gpt4_train)
val_dataset = JsonInstructionDataset(json_gpt4_val)

In [ ]:
print(train_dataset[0])

{'input_ids': tensor([21017, 46486,    25,   198, 23318,  1115,  9040,   329, 10589,  5448,
           13,   198,   198, 21017, 23412,    25,   628,   198, 21017, 18261,
           25,   198,    16,    13, 27574,   257, 12974,   290, 48102,  5496,
           25,  6889,  1654,   534, 13840,   389, 19889,   286,   257,  4996,
          286, 15921,   290, 13701,    11, 10904,  7532,    11,  2187, 21824,
           11,   290,  5448, 27997,    13,   770,  5419,   284,  2148,   534,
         1767,   351,   262,  6393, 20901,   284,  2163,   379,   663,  1266,
          290,   460,  1037,  2948, 10726, 10040,    13,   198,   198,    17,
           13,  1985,   496,   287,  3218,  3518,  3842,    25, 32900,   318,
         8780,   329, 10941,  1913, 11945,    11, 12749,    11,   290, 21134,
         1535,    13, 36223,   329,   379,  1551,  6640,  2431,   286, 10768,
        43294,  5517,   393,  5441,  2431,   286, 31543,  5517,  1123,  1285,
           13,   198,   198,    18,    13,  3497, 

In [ ]:
from transformers import PretrainedConfig

class GPTConfig(PretrainedConfig):
    def __init__(self, **kwargs):
        # config
        self.vocab_size = 50257
        self.context_length = 1024
        self.emb_dim = 1280
        self.n_heads = 20
        self.n_layers = 36
        self.drop_rate = 0.1
        self.qkv_bias = True
        super().__init__(**kwargs)

# attaching config to model
model.config = GPTConfig()

In [ ]:
from transformers.modeling_outputs import CausalLMOutput
import types

def hf_forward(self, input_ids=None, labels=None, attention_mask=None, **kwargs):

    batch_size, seq_len = input_ids.shape
    tok_embeds = self.tok_emb(input_ids)
    pos_embeds = self.pos_emb(torch.arange(seq_len, device=input_ids.device))
    x = tok_embeds + pos_embeds
    x = self.drop_emb(x)
    x = self.trf_blocks(x)
    x = self.final_norm(x)
    logits = self.out_head(x)

    loss = None
    if labels is not None:
        loss_fct = torch.nn.CrossEntropyLoss()
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))

    return CausalLMOutput(
        loss=loss,
        logits=logits,
        hidden_states=None,
        attentions=None
    )

# applying the patch
model.forward = types.MethodType(hf_forward, model)

Fixed forward method - using CausalLMOutput


In [ ]:
def prepare_inputs_for_generation(self, input_ids, **kwargs):
    return {"input_ids": input_ids, **kwargs}

if not hasattr(model, 'prepare_inputs_for_generation'):
    model.prepare_inputs_for_generation = prepare_inputs_for_generation.__get__(model, type(model))

In [ ]:
from peft import LoraConfig, get_peft_model


def setup_lora_model(model):
    target_modules = [
        "W_query", "W_key", "W_value", "out_proj",
        "ff.layers.0", "ff.layers.2",
        "out_head"
    ]

    # LoRA configuration
    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=target_modules,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        modules_to_save=None
    )

    # applying LoRA
    model = get_peft_model(model, lora_config)

    return model

# applying LoRA to model
model = setup_lora_model(model)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:73: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:196: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
import transformers

transformers.logging.set_verbosity_info()

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-5,
    fp16=True,

    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,

    save_strategy="no",
    report_to="none",

    dataloader_pin_memory=False,
    dataloader_drop_last=True,
    remove_unused_columns=True,

    # optimizations for T4 GPU
    torch_compile=False,
    fp16_full_eval=False,

    # learning rate settings
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="adamw_torch_fused",

    # progress bar settings
    disable_tqdm=False,
    eval_delay=0
)

PyTorch: setting up devices


In [ ]:
def compute_metrics(eval_pred):
    return {}

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

Using auto half precision backend


In [ ]:
os.environ["WANDB_DISABLED"] = "true"

gc.collect()
torch.cuda.empty_cache

print("Started training...")
trainer.train()
print("Training completed!")

***** Running training *****
  Num examples = 5,000
  Num Epochs = 1
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 16
  Gradient Accumulation steps = 8
  Total optimization steps = 313


Started training...


  Number of trainable parameters = 7,047,816


OutOfMemoryError: CUDA out of memory. Tried to allocate 40.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 36.12 MiB is free. Process 20166 has 14.70 GiB memory in use. Of the allocated memory 14.57 GiB is allocated by PyTorch, and 5.64 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
# def test_model(model, instruction, input_text="", max_new_tokens=1024):
#     # Format the prompt like during training
#     if input_text:
#         prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
#     else:
#         prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

#     # Tokenize
#     encoding = tiktoken.get_encoding('gpt2')
#     input_ids = encoding.encode(prompt)
#     input_tensor = torch.tensor([input_ids]).to(device)

#     # Manual generation (no .generate() method)
#     model.eval()
#     with torch.no_grad():
#         generated = input_tensor

#         for i in range(max_new_tokens):
#             # Forward pass
#             outputs = model(input_ids=generated)
#             logits = outputs['logits']

#             # Get last token logits
#             next_token_logits = logits[0, -1, :]

#             # Apply temperature and sample
#             next_token_logits = next_token_logits / 0.7  # temperature
#             probs = torch.softmax(next_token_logits, dim=-1)
#             next_token = torch.multinomial(probs, num_samples=1)

#             # Stop if EOT token is generated BEFORE appending
#             if next_token.item() == encoding.eot_token:
#                 break

#             # Append to generated sequence
#             generated = torch.cat([generated, next_token.unsqueeze(0)], dim=1)

#     # Decode
#     response = encoding.decode(generated[0].tolist())
#     # Extract only the response part (after "### Response:\n")
#     response_text = response.split("### Response:\n")[-1]

#     # Remove endoftext token if it exists and clean up
#     response_text = response_text.replace('<|endoftext|>', '').strip()

#     return response_text

# # Test examples
# print("🧪 Testing the fine-tuned model:\n")

# # Example 1: General instruction
# test1 = test_model(
#     model,
#     instruction="what is AI?",
#     input_text="",
# )
# print(f"Test 1 - Explanation:\n{test1}\n")

In [ ]:
# # After instruction fine-tuning, merge LoRA into base model
# merged_model = model.merge_and_unload()

# # Save the MERGED weights (now base model has instruction knowledge)
# torch.save(merged_model.state_dict(), "/content/drive/MyDrive/llm_from_scratch/text_generation_model/instruction_model/instruction_tuned_model_weights.pth")

# print("✅ Saved unified model with instruction knowledge")